# Model Comparison: Quality vs. Cost Across OpenAI Models

Based on: [KAMI: Kamiwaza Agentic Merit Index](https://arxiv.org/abs/2511.08042) (Nov 2025)

## The Problem

GPT-4o is powerful but costs $2.50/$10 per 1M tokens. GPT-4o-mini costs $0.15/$0.60. GPT-4.1-nano costs $0.10/$0.40. **Which model gives you the best quality for your budget?**

## The Quality/$ Metric

Raw quality scores and raw costs are not enough to make a decision. A model scoring 0.95 quality at $0.005/query is not necessarily better than one scoring 0.85 at $0.0003/query — it depends on your volume and budget.

**Quality/$** (quality per dollar) normalizes this by dividing the quality score by the cost:

```
Quality/$ = quality_score / cost_per_query
```

| Model | Quality | Cost/Query | Quality/$ | Interpretation |
|-------|:-------:|:----------:|:---------:|---------------|
| GPT-4o | 0.95 | $0.005 | 190 | Best quality, but lowest value per dollar |
| GPT-4o-mini | 0.85 | $0.0003 | 2,833 | Slightly lower quality, but 15x better value |
| GPT-4.1-nano | 0.80 | $0.0002 | 4,000 | Lowest quality, but highest value per dollar |

A high Quality/$ means you get more quality per dollar spent. For high-volume applications (thousands of queries per day), the model with the highest Quality/$ is usually the right choice, as long as the raw quality meets your minimum threshold.

## What We Compare

We run the same evaluation task on 3 OpenAI models and compare quality (OutputEvaluator score) vs. cost (token usage x pricing). The same LLM judge evaluates all models for fairness.

> **What to look for:** The final comparison table shows Quality, Cost, Tokens, and Quality/$ for each model. The model with the highest Quality/$ is not always the most expensive one — often a mid-tier model delivers the best value.

In [ ]:
# %pip install strands-agents strands-agents-evals boto3

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Fix for Jupyter async event loop

from strands import Agent, tool
from strands.models.openai import OpenAIModel
from strands_evals import Experiment, Case
from strands_evals.evaluators import OutputEvaluator

# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
MODELS = {
    "gpt-4o": {"id": "gpt-4o", "input_price": 2.50, "output_price": 10.00},
    "gpt-4o-mini": {"id": "gpt-4o-mini", "input_price": 0.15, "output_price": 0.60},
    "gpt-4.1-nano": {"id": "gpt-4.1-nano", "input_price": 0.10, "output_price": 0.40},
}

JUDGE_MODEL = OpenAIModel(model_id="gpt-4o-mini")  # Same judge for all

@tool
def search_flights(origin: str, destination: str, date: str) -> str:
    """Search for available flights."""
    return f"Flights {origin}->{destination} on {date}: BA117 $450, DL1 $520, UA100 $480"

QUERY = "Find flights from NYC to London for next Friday"
RUBRIC = "Rate 0-1. 0.8+: Specific flights with details. 0.4-0.7: Partial info. 0.0-0.3: Vague."

evaluator = OutputEvaluator(rubric=RUBRIC, model=JUDGE_MODEL)

print("=" * 70)
print("MODEL COMPARISON: Quality vs Cost")
print("=" * 70)

results = {}
for name, config in MODELS.items():
    agent = Agent(
        model=OpenAIModel(model_id=config["id"]),
        tools=[search_flights],
        system_prompt="You are a travel assistant. Use tools to answer.",
    )
    result = agent(QUERY)
    response = str(result)

    # Get cost
    m = result.metrics
    input_t = m.accumulated_usage.get("inputTokens", 0)
    output_t = m.accumulated_usage.get("outputTokens", 0)
    cost = (input_t * config["input_price"] + output_t * config["output_price"]) / 1_000_000

    # Get quality score
    case = Case(name=name, input=QUERY, expected_output="Specific flights")
    exp = Experiment(cases=[case], evaluators=[evaluator])
    reports = exp.run_evaluations(lambda c, r=response: r)
    quality = reports[0].overall_score

    results[name] = {"quality": quality, "cost": cost, "tokens": input_t + output_t}
    print(f"\n  {name}:")
    print(f"    Quality: {quality:.2f} | Cost: ${cost:.5f} | Tokens: {input_t + output_t:,}")

# Comparison table
print(f"\n{'=' * 70}")
print(f"{'Model':<18} {'Quality':<10} {'Cost':<12} {'Tokens':<10} {'Quality/$':<10}")
print(f"{'-'*18} {'-'*10} {'-'*12} {'-'*10} {'-'*10}")
for name, r in results.items():
    q_per_dollar = r["quality"] / r["cost"] if r["cost"] > 0 else 0
    print(f"  {name:<16} {r['quality']:<10.2f} ${r['cost']:<11.5f} {r['tokens']:<10,} {q_per_dollar:<10.0f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
fig.set_facecolor('white')

colors = ['#2196F3', '#4CAF50', '#FF9800']

for i, (name, r) in enumerate(results.items()):
    ax.scatter(r["tokens"], r["quality"], s=200, color=colors[i], zorder=5, edgecolors='white', linewidth=2)
    ax.annotate(name, (r["tokens"], r["quality"]),
                textcoords="offset points", xytext=(10, 10),
                fontsize=12, fontweight='bold', color=colors[i])

ax.set_xlabel('Total Tokens Used', fontsize=12)
ax.set_ylabel('Quality Score', fontsize=12)
ax.set_title('Model Comparison: Quality vs Tokens\n(Up and left is better)', fontweight='bold', fontsize=14)
ax.set_ylim(0, 1.1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## Key Insight

**Quality/$ is the metric that matters for model selection at scale.** A model scoring 0.85 at $0.001/query may be better than one scoring 0.95 at $0.01/query for high-volume applications. The "best" model depends on your quality threshold and query volume.

**How to use these results:**
1. Define your minimum acceptable quality (e.g., 0.80)
2. Filter to models that meet the threshold
3. Among those, pick the one with the highest Quality/$ (lowest cost)

> **What to look for:** If all three models score above your quality threshold, the cheapest one is the right choice. If only the expensive model meets the threshold, you pay the premium. The Quality/$ column quantifies the trade-off.

**Next:** [Demo 03 - Cost-Quality Pareto](../03-cost-quality-pareto/) — Find the optimal model for your quality threshold and budget.